#DKG_Visualizaiton
#Semantic Graph Visualization 

This notebook provides an adjustable framework to visualize the Resource Description Framework (RDF) Knowledge Graph. It facilitates the visual inspection of the network topology by allowing the isolation of specific consumer sub-graphs and the filtration of high-frequency telemetric data to enhance structural clarity.

In [ ]:
import os
import networkx as nx
import matplotlib.pyplot as plt
from rdflib import Graph


# FILE PATH CONFIGURATION

# Specify the absolute or relative path to semantic database (.ttl)
# Example: r"C:\Users\Name\FileName\dynamic_grid.ttl"
GRAPH_FILE_PATH = "dynamic_grid.ttl"

In [ ]:
#visualization algorithm
def visualize_knowledge_graph(file_path, target_consumer=None, exclude_measurements=False):
    """
    visualizes a directed graph representation of the semantic database.
    
    Parameters:
    - file_path (str): The directory path to the .ttl file.
    - target_consumer (str): Isolates the sub-graph related to a specific consumer (e.g., "H1").
    - exclude_measurements (bool): Omits transient operational data to clarify the static topology.
    """
    
    # Initialize and load the Knowledge Graph
    dkg = Graph()
    if os.path.exists(file_path):
        dkg.parse(file_path, format="turtle")
    else:
        print(f"Error: The designated file '{file_path}' was not located.")
        return

    # Initialize the NetworkX directed graph structure
    nx_graph = nx.DiGraph()
    
    # Process and apply filtering conditions to semantic triplets
    for subject, predicate, obj in dkg:
        s_str = str(subject).split("/")[-1]
        p_str = str(predicate).split("/")[-1]
        o_str = str(obj).split("/")[-1] if not isinstance(obj, str) else str(obj)
        
        # Apply operational data filtration
        if exclude_measurements and ("Meas_" in s_str or p_str in ["hasTime", "hasVoltage", "hasPower", "hasActivePower", "hasReactivePower"]):
            continue
            
        # Apply consumer isolation
        if target_consumer and target_consumer not in s_str and target_consumer not in o_str:
            continue
                
        nx_graph.add_edge(s_str, o_str, label=p_str)

    if len(nx_graph.nodes) == 0:
        print("The current filtering parameters yielded an empty graph. Adjust your configuration.")
        return

    # Compute structural layout and visualize
    plt.figure(figsize=(14, 10))
    pos = nx.spring_layout(nx_graph, k=0.6, iterations=50)
    
    nx.draw(nx_graph, pos, 
            with_labels=True, 
            node_color='#add8e6', 
            node_size=2500, 
            font_size=9, 
            font_weight='bold', 
            edge_color='#888888',
            arrows=True,
            arrowsize=15)
            
    edge_labels = nx.get_edge_attributes(nx_graph, 'label')
    nx.draw_networkx_edge_labels(nx_graph, pos, edge_labels=edge_labels, font_size=8, font_color='red')
    
    # Dynamically generate the plot title based on selected parameters
    title = "Semantic Knowledge Graph Visualization"
    if target_consumer:
        title += f" (Isolated Consumer: {target_consumer})"
    if exclude_measurements:
        title += " [Topological Assets Only]"
        
    plt.title(title, fontsize=14, fontweight='bold')
    plt.axis('off')
    plt.tight_layout()
    plt.show()

In [ ]:
# EXECUTION CONFIGURATION

# Modify the parameters below to adjust the graphical output.

TARGET_CONSUMER = None         # Set to a string (e.g., "H1") to isolate, or None for the full network.
EXCLUDE_MEASUREMENTS = False   # Set to True to hide readings and view only the asset topology.

# Execute the visualizeing function
visualize_knowledge_graph(
    file_path=GRAPH_FILE_PATH, 
    target_consumer=TARGET_CONSUMER, 
    exclude_measurements=EXCLUDE_MEASUREMENTS
)